# ISU Campus Assistant — Hugging Face Space edition (ZeroGPU)

This notebook puts ReggieBot on a **Hugging Face Space** with **ZeroGPU**, Hugging Face's free shared GPUs.
The result is a permanent public address (`huggingface.co/spaces/<you>/reggiebot`) that works when this PC is
off.

ZeroGPU cannot run Ollama, so this edition loads the model **inside the app with transformers** instead.
Everything else is the same `reggiebot` package the Ollama notebook uses: the search, the prompt, the
fallbacks and the chat page. Only the model plugs in differently, through the same `check()` / `warm_up()` /
`generate()` interface as the Ollama and fake models.

| | Ollama edition (this PC) | Space edition (ZeroGPU) |
|---|---|---|
| Where the model runs | the Ollama app, a separate program | inside the app, loaded with transformers |
| Model | `gemma4:12b`, compressed download (~8 GB) | `google/gemma-4-12B-it`, full bfloat16 weights (~24 GB) |
| GPU | your graphics card, all the time | borrowed for each answer (48 GB), then given back |
| Address | `gradio.live`, a new one every start | permanent: `huggingface.co/spaces/<you>/reggiebot` |
| Runs when | this PC, Ollama and the app are on | always; a Space that sleeps wakes up on the next visit, which takes a few minutes |
| Cost | electricity | free to host (2 ZeroGPU Spaces per free account); visitors spend their own daily GPU time |
| Code | `ISU-CampusAssistant-RAG-Ollama.ipynb`, `app.py` | `space_app.py`, `reggiebot/hf_model.py` |

Nothing in the Ollama edition changes. This notebook and three new files make up the Space edition:

* `reggiebot/hf_model.py`: `TransformersModel`, the transformers version of the model plug
* `space_app.py`: the Space's entry point, which loads the model at startup and answers inside `@spaces.GPU`
* `space_build/`: made by §3, the folder that gets uploaded to the Space

### What ZeroGPU requires

| Requirement | Details |
|---|---|
| A Hugging Face account | free; to host ZeroGPU Spaces it needs a verified email and must be older than 30 days (up to 2 Spaces). PRO hosts up to 10 |
| Gradio | ZeroGPU only works with Gradio Spaces, which ReggieBot already is |
| The model on `cuda` at startup | `space_app.py` loads it at module level, as ZeroGPU requires |
| GPU work inside `@spaces.GPU` | each answer borrows a GPU for up to `GPU_SECONDS` (60 s; an answer takes a few) |
| Visitors' daily GPU time | 2 minutes without an account, 5 with a free account, 40 with PRO. At a few seconds per answer that is roughly 25–100 questions per visitor per day |

---
## Settings

In [1]:
import os
import shutil
from pathlib import Path

# The model the Space loads. Gemma 4 12B needs ~24 GB of GPU memory: fine on ZeroGPU (48 GB), too big for most
# PCs, so the local test below uses a tiny model instead. The wiring is identical; only the answers are worse.
SPACE_MODEL_ID   = "google/gemma-4-12B-it"
LOCAL_TEST_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"   # ~270 MB, runs on a CPU
GPU_SECONDS      = 60                                      # GPU time allowed per answer on ZeroGPU

BUILD_DIR     = Path("space_build")    # the folder that gets uploaded to the Space (git ignores it)
SPACE_REPO    = "JakoBob/Campus_Assistant"                   # e.g. "your-hf-username/reggiebot"
SPACE_PRIVATE = False                   # start private; make it public from the Space's settings when ready

---
## 1 · Try the transformers model on this computer

The same program the Space will run, here with a tiny model so it fits on any PC: the real search over the
saved data, the real prompt, `TransformersModel` streaming the answer, and the `Answer` with its status and
timing. The tiny model is ~90 times smaller than Gemma 4 12B, so expect weak answers; this checks the wiring,
not the quality. The first run downloads it (~270 MB).

In [2]:
RUN_LOCAL_TEST = True  #@param {type:"boolean"}

from reggiebot.answer import Answerer, SYSTEM_PROMPT
from reggiebot.config import CACHE_DIR, CHROMA_DIR
from reggiebot.core import CatalogApp
from reggiebot.hf_model import TransformersModel

if RUN_LOCAL_TEST:
    LOCAL_MODEL = TransformersModel(LOCAL_TEST_MODEL, max_new_tokens=300)
    APP = CatalogApp()
    APP.answerer = Answerer(APP.index, model=LOCAL_MODEL)
    APP.load(progress=lambda msg: print("  ", msg))
    result = APP.ask("When will I get my refund?")
    print("\n", repr(result))
else:
    print("Local test skipped - tick RUN_LOCAL_TEST to run it.")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] using HuggingFaceTB/SmolLM2-135M-Instruct via transformers on cpu (bfloat16, loaded in 1.4 s)
   loaded 6783 saved documents from C:\Users\Jake\Downloads\ISU-RAG-VSCode\isu_catalog_cache - no crawling
   building keyword index
   vector DB already current - 6783 vectors
   index ready - 6783 courses
The first refund process for each semester starts the Tuesday before classes start (this is an internal process and refunds are not yet available). Students with eRefunds will receive their refund the Friday that refunds are run. Paper refund checks will be mailed to the student's Mailing address the Tuesday after refunds are run. Sign up for eRefunds to receive your refund faster! The first refund process for each semester starts the Tuesday before classes start (this is an internal process and refunds are not yet available).

[9] Student Accounts | Illinois State - Refunds
Section: Billing
Page section: Refunds
Source: https://studentaccounts.illinoisstate.edu/index.php
Las

---
## 2 · Check Gemma 4's prompt (no download of the weights)

This downloads only Gemma 4's configuration and tokenizer (a few MB), then shows the exact text the model will
read. Two things to look for: the system message (ReggieBot's rules) is kept, and with `enable_thinking=False`
the template pre-fills an empty "thought" channel, so the model answers directly instead of reasoning first.

In [3]:
from transformers import AutoConfig, AutoModelForMultimodalLM, AutoTokenizer

config = AutoConfig.from_pretrained(SPACE_MODEL_ID)
print("transformers loads it as:", AutoModelForMultimodalLM._model_mapping[type(config)].__name__, "\n")
tokenizer = AutoTokenizer.from_pretrained(SPACE_MODEL_ID)
messages = [{"role": "system", "content": SYSTEM_PROMPT.splitlines()[0] + " (... the rest of the rules)"},
            {"role": "user", "content": "CONTEXT - Illinois State University catalog excerpts\n[1] ...\n\n"
                                        "QUESTION\nWhat are the prerequisites for IT 214?"}]
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False))

transformers loads it as: Gemma4UnifiedForConditionalGeneration 

<bos><|turn>system
You are the Illinois State University course catalog assistant. (... the rest of the rules)<turn|>
<|turn>user
CONTEXT - Illinois State University catalog excerpts
[1] ...

QUESTION
What are the prerequisites for IT 214?<turn|>
<|turn>model
<|channel>thought
<channel|>


---
## 3 · Build the Space folder

A Space is a small git repository. This cell assembles everything it needs in `space_build/`:

| In the folder | What it is |
|---|---|
| `space_app.py` | the entry point the Space runs |
| `reggiebot/` | the package, copied as it is now |
| `isu_catalog_cache/` | the saved data: `corpus.jsonl`, `snapshot.json` and the Chroma database (~67 MB) |
| `requirements.txt` | what the Space installs, pinned to the versions tested here |
| `README.md` | the Space's settings (Gradio version, Python 3.12, `app_file: space_app.py`) and a short description |

Re-run it after changing the code or refreshing the data, then upload again (§4).

In [4]:
import gradio

SPACE_REQUIREMENTS = """\
# Installed on the Space. Gradio itself comes from sdk_version in README.md.
spaces                        # ZeroGPU's @spaces.GPU
torch==2.13.0                 # ZeroGPU supports PyTorch 2.8 to 2.13
transformers==5.19.0          # knows Gemma 4 (gemma4_unified); the version tested in this notebook
sentence-transformers==6.1.0  # MiniLM embeddings for the search (on the CPU)
chromadb==1.5.9               # the version that built the saved vector database
rank-bm25>=0.2.2
requests>=2.31
beautifulsoup4>=4.12
lxml>=5.0
trafilatura>=1.6
numpy>=1.26
scikit-learn>=1.3
python-dotenv>=1.0
huggingface_hub
"""

SPACE_README = f"""---
title: ReggieBot
emoji: 🐦
colorFrom: red
colorTo: gray
sdk: gradio
sdk_version: {gradio.__version__}
python_version: "3.12"
app_file: space_app.py
pinned: false
short_description: Student RAG assistant for ISU courses and campus info
---

# ReggieBot - ISU Campus Assistant

Ask about Illinois State University courses, billing, housing and dining. Answers come only from ISU's own
catalog and websites, with a link to every source, written by {SPACE_MODEL_ID} on ZeroGPU.

This is a student project, not an official Illinois State University service. Answers can be wrong; check
anything important with the university.
"""


def build_space():
    """Assemble space_build/ from the current code and data. Returns the folder."""
    if BUILD_DIR.exists():
        if any(BUILD_DIR.iterdir()) and not (BUILD_DIR / "space_app.py").exists():
            raise RuntimeError(f"{BUILD_DIR} exists and is not a Space build folder - not deleting it.")
        shutil.rmtree(BUILD_DIR)
    BUILD_DIR.mkdir()
    shutil.copy2("space_app.py", BUILD_DIR / "space_app.py")
    shutil.copytree("reggiebot", BUILD_DIR / "reggiebot", ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
    data = BUILD_DIR / "isu_catalog_cache"
    data.mkdir()
    for name in ("corpus.jsonl", "snapshot.json"):
        shutil.copy2(CACHE_DIR / name, data / name)
    shutil.copytree(CHROMA_DIR, data / "chroma")
    (BUILD_DIR / "requirements.txt").write_text(SPACE_REQUIREMENTS, encoding="utf-8")
    (BUILD_DIR / "README.md").write_text(SPACE_README, encoding="utf-8")
    return BUILD_DIR


build_space()
total = 0
for path in sorted(BUILD_DIR.rglob("*")):
    if path.is_file():
        total += path.stat().st_size
        if path.stat().st_size > 1_000_000 or path.parent == BUILD_DIR:
            print(f"{path.relative_to(BUILD_DIR).as_posix():<55} {path.stat().st_size / 1e6:8.1f} MB")
print(f"{'total':<55} {total / 1e6:8.1f} MB in {sum(1 for p in BUILD_DIR.rglob('*') if p.is_file())} files")

isu_catalog_cache/chroma/adaab3fb-acf4-4fda-91c9-590ccc2e5980/data_level0.bin     11.4 MB
isu_catalog_cache/chroma/chroma.sqlite3                     53.4 MB
isu_catalog_cache/corpus.jsonl                               9.8 MB
README.md                                                    0.0 MB
requirements.txt                                             0.0 MB
space_app.py                                                 0.0 MB
total                                                       75.3 MB in 19 files


---
## 4 · Upload to Hugging Face (you run this)

1. **Account:** a free Hugging Face account with a verified email, older than 30 days.
2. **Token:** create a **write** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
   and put it in `.env` as `HF_TOKEN=...`. Restart the kernel.
3. **Name:** set `SPACE_REPO = "your-username/reggiebot"` in the settings cell.
4. **Upload:** tick `UPLOAD_SPACE` and run the cell below. It creates the Space (private), sets the model,
   asks for ZeroGPU hardware and uploads `space_build/`.
5. **Hardware:** if the cell says ZeroGPU could not be requested, open the Space's **Settings → Hardware** and
   pick **ZeroGPU**. Without it, the Space would try to load a 24 GB model on a small CPU machine and fail.
6. **Wait:** the first start installs the packages and downloads Gemma 4 (~24 GB). That takes several minutes.
   **Logs** on the Space's page shows progress, ending with `loaded: google/gemma-4-12B-it ...`.

Before making it public, check with your instructor: it uses ISU's name and colors.

In [5]:
UPLOAD_SPACE = True  #@param {type:"boolean"}

if not UPLOAD_SPACE:
    print("Upload skipped - set SPACE_REPO above, tick UPLOAD_SPACE and run this cell.")
elif not SPACE_REPO:
    print('Set SPACE_REPO = "your-hf-username/reggiebot" in the settings cell first.')
elif not os.environ.get("HF_TOKEN"):
    print("No HF_TOKEN found. Add HF_TOKEN=<a write token from huggingface.co/settings/tokens> to .env "
          "and restart the kernel.")
elif not (BUILD_DIR / "space_app.py").exists():
    print("Build the Space folder first (§3).")
else:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo(SPACE_REPO, repo_type="space", space_sdk="gradio", private=SPACE_PRIVATE, exist_ok=True)
    api.add_space_variable(SPACE_REPO, "SPACE_MODEL_ID", SPACE_MODEL_ID)
    api.add_space_variable(SPACE_REPO, "SPACE_GPU_SECONDS", str(GPU_SECONDS))
    try:
        api.request_space_hardware(SPACE_REPO, "zero-a10g")      # Hugging Face's id for ZeroGPU hardware
        print("ZeroGPU requested.")
    except Exception as exc:
        print(f"Could not request ZeroGPU from here ({type(exc).__name__}: {str(exc)[:120]}).")
        print(f"Pick it by hand: https://huggingface.co/spaces/{SPACE_REPO}/settings -> Hardware -> ZeroGPU")
    api.upload_folder(folder_path=str(BUILD_DIR), repo_id=SPACE_REPO, repo_type="space",
                      commit_message="Update ReggieBot")
    print(f"Uploaded. Watch it start: https://huggingface.co/spaces/{SPACE_REPO} (open Logs)")

ZeroGPU requested.
Uploaded. Watch it start: https://huggingface.co/spaces/JakoBob/Campus_Assistant (open Logs)


---
## 5 · Once it is running

* **Check it the way the Ollama edition was checked:** ask the starter questions, one trap question
  ("What is my current account balance?") and one billing question, and confirm the source links. The
  status line under the header should read `RAG, google/gemma-4-12B-it`.
* **Speed:** the first answer after a quiet period waits for a GPU to be assigned; after that, answers should
  take a few seconds. If one fails, the chat shows the catalog matches and says the model is busy or the free
  GPU time ran out.
* **Answers can differ slightly** from the Ollama edition: the Space runs Gemma 4 at full precision, while
  Ollama's download is compressed. The search and the prompt are identical.
* **The data is a snapshot.** The Space answers from the data copied in §3. "Check catalog for updates" does
  crawl from the Space, but the Space's disk resets on every restart, so refresh here, rebuild (§3) and upload
  (§4) instead. The same goes for the usage log.
* **To update** after changing code: re-run §3 and §4. The Space restarts with the new files.
* **To stop it:** pause the Space in its **Settings**, or make it private.